In [4]:
import os
import time
import pandas as pd
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types

In [5]:
spark = (
    SparkSession.builder
        .appName("CryptoETL")
        .config("spark.master", "spark://spark-master:7077")
        # ---- Iceberg + Hive Catalog ----
        .config("spark.sql.catalog.hive_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.hive_catalog.catalog-impl", "org.apache.iceberg.hive.HiveCatalog")
        .config("spark.sql.catalog.hive_catalog.uri", "thrift://hive-metastore:9083")
        .config("spark.sql.catalog.hive_catalog.warehouse", "s3a://crypto-data-lake/")
        # ---- Default catalog
        .config("spark.sql.defaultCatalog", "hive_catalog")
        # ---- S3 (MinIO) ----
        .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        # ---- Iceberg Extensions ----
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
        # ---- Extra JARs ----
        .config("spark.jars", ",".join([
            "/opt/spark-extra-jars/iceberg-spark-runtime-3.5_2.12-1.6.1.jar",
            "/opt/spark-extra-jars/hadoop-aws-3.3.4.jar",
            "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.262.jar"
        ]))
        .getOrCreate()
)

25/09/28 11:42:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [6]:
df_sorted = (
    spark.sql("select * from serving_db.klines")
    .coalesce(1) # one partition, not shuffle
    .sortWithinPartitions("group_id")
)

25/09/28 11:42:16 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [7]:
df_sorted.show()

+--------+-------------------+----------------+----------+----------+---------+-----------+--------+----------------+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price|  volume|      close_time|
+--------+-------------------+----------------+----------+----------+---------+-----------+--------+----------------+
| 1948896|2025-08-01 00:00:00|1754006400328945| 115764.07| 115829.46|115308.55|  115313.01| 302.159|1754007299467573|
| 1948897|2025-08-01 00:15:00|1754007300010950| 115313.01|  115933.0| 115313.0|  115800.01| 450.539|1754008199447993|
| 1948898|2025-08-01 00:30:00|1754008200077603|  115800.0|  115800.0|115423.87|  115517.98| 184.425|1754009099900832|
| 1948899|2025-08-01 00:45:00|1754009100223687| 115517.99| 115527.53|114313.13|  115427.27|1589.765|1754009999974074|
| 1948900|2025-08-01 01:00:00|1754010000363342| 115427.27| 115609.99| 114600.0|   114649.9| 681.883|1754010899995166|
| 1948901|2025-08-01 01:15:00|1754010900041356|  114649.

In [8]:
schema = types.StructType([
    *df_sorted.schema.fields,  # keep all original fields
    types.StructField("ema7", types.DoubleType(), True),
    types.StructField("ema20", types.DoubleType(), True)
])

In [9]:
def round_half_up(x, decimals=2):
    if x is None:
        return None
    factor = 10 ** decimals
    return float(int(x * factor + 0.5)) / factor

def calc_ema(value, state):
    if value is None:
        return None
    prev, buffer, period, k = state["prev"], state["buffer"], state["period"], state["k"]
    if prev is None:
        buffer.append(value)
        if len(buffer) == period:
            ema = sum(buffer) / len(buffer)
        else:
            ema = None
    else:
        ema = (value - prev) * k + prev

    state["prev"] = ema
    return ema

def rounded(dec):
    return float(dec.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))

def ema_in_chunks(iterator):
    ema_configs = {
        "ema7": {"period": 7, "k": 2 / (7 + 1), "prev": None, "buffer": []},
        "ema20": {"period": 20, "k": 2 / (20 + 1), "prev": None, "buffer": []}
    }

    for pdf in iterator:
        ema7, ema20 = [], []
        for p in pdf["close_price"]:
            price = float(p)
            # step 1: compute ema12, ema26
            e7 = calc_ema(price, ema_configs["ema7"])
            ema7.append(round_half_up(e7, 2) if e7 is not None else None)
            e20 = calc_ema(price, ema_configs["ema20"])
            ema20.append(round_half_up(e20, 2) if e20 is not None else None)
            
        pdf["ema7"] = ema7
        pdf["ema20"] = ema20
        # final order
        pdf = pdf[[*pdf.columns[:-2], "ema7", "ema20"]]
        yield pdf

In [10]:
df = df_sorted.mapInPandas(ema_in_chunks, schema)

In [11]:
df.createOrReplaceTempView("temp")

In [16]:
df = spark.sql("""
with cte as (
    select
        *,
        case 
            when ema7 > ema20 then 'uptrend' 
            when ema7 < ema20 then 'downtrend' 
            else NULL 
        end as trend,
        lag(open_price, 2) over(order by group_id) as o1,
        lag(close_price, 2) over(order by group_id) as c1,
        lag(open_price, 1) over(order by group_id) as o2,
        lag(close_price, 1) over(order by group_id) as c2,
        open_price as o3,
        close_price as c3,
        lag(high_price, 1) over(order by group_id) as h2,
        lag(low_price, 1) over(order by group_id) as l2
    from temp
)
select
    group_id,
    group_date,
    close_price,
    trend,
    case 
        when c1 < o1
             and abs(c2 - o2) <= 0.5 * abs(o1 - c1)
             and c2 < c1 and o2 < c1
             and c3 > o3
             and c3 >= (o1 + c1)/2
             and trend = 'downtrend'
        then 'morning star'

        when c1 > o1
             and abs(c2 - o2) <= 0.5 * abs(c1 - o1)
             and o2 > c1 and c2 > c1
             and c3 < o3
             and c3 <= (o1 + c1)/2
             and trend = 'uptrend'
        then 'evening star'

        when c1 < o1
             and abs(c2 - o2) <= 0.1 * (h2 - l2)
             and c2 < c1 and o2 < c1
             and c3 > o3
             and c3 >= (o1 + c1)/2
             and trend = 'downtrend'
        then 'morning doji star'

        when c1 > o1
             and abs(c2 - o2) <= 0.1 * (h2 - l2)
             and o2 > c1 and c2 > c1
             and c3 < o3
             and c3 <= (o1 + c1)/2
             and trend = 'uptrend'
        then 'evening doji star'

        else null
    end as pattern
from cte
""")

In [17]:
df.show(100)

25/09/28 11:53:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/09/28 11:53:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/09/28 11:53:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+--------+-------------------+-----------+---------+-------+
|group_id|         group_date|close_price|    trend|pattern|
+--------+-------------------+-----------+---------+-------+
| 1948896|2025-08-01 00:00:00|  115313.01|     NULL|   NULL|
| 1948897|2025-08-01 00:15:00|  115800.01|     NULL|   NULL|
| 1948898|2025-08-01 00:30:00|  115517.98|     NULL|   NULL|
| 1948899|2025-08-01 00:45:00|  115427.27|     NULL|   NULL|
| 1948900|2025-08-01 01:00:00|   114649.9|     NULL|   NULL|
| 1948901|2025-08-01 01:15:00|  115190.38|     NULL|   NULL|
| 1948902|2025-08-01 01:30:00|  115296.45|     NULL|   NULL|
| 1948903|2025-08-01 01:45:00|  115331.86|     NULL|   NULL|
| 1948904|2025-08-01 02:00:00|   115600.0|     NULL|   NULL|
| 1948905|2025-08-01 02:15:00|  115619.94|     NULL|   NULL|
| 1948906|2025-08-01 02:30:00|  115900.01|     NULL|   NULL|
| 1948907|2025-08-01 02:45:00|  115966.12|     NULL|   NULL|
| 1948908|2025-08-01 03:00:00|   115816.0|     NULL|   NULL|
| 1948909|2025-08-01 03:

In [18]:
spark.sql("""
drop table if exists serving_db.pattern_three
""")

DataFrame[]

In [19]:
df.writeTo("serving_db.pattern_three").tableProperty("format-version", "2").createOrReplace()

25/09/28 11:54:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/09/28 11:54:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/09/28 11:54:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [20]:
spark.sql("""
select * from serving_db.pattern_three
""").orderBy("group_id", ascending=True).show(100)

+--------+-------------------+-----------+---------+-------+
|group_id|         group_date|close_price|    trend|pattern|
+--------+-------------------+-----------+---------+-------+
| 1948896|2025-08-01 00:00:00|  115313.01|     NULL|   NULL|
| 1948897|2025-08-01 00:15:00|  115800.01|     NULL|   NULL|
| 1948898|2025-08-01 00:30:00|  115517.98|     NULL|   NULL|
| 1948899|2025-08-01 00:45:00|  115427.27|     NULL|   NULL|
| 1948900|2025-08-01 01:00:00|   114649.9|     NULL|   NULL|
| 1948901|2025-08-01 01:15:00|  115190.38|     NULL|   NULL|
| 1948902|2025-08-01 01:30:00|  115296.45|     NULL|   NULL|
| 1948903|2025-08-01 01:45:00|  115331.86|     NULL|   NULL|
| 1948904|2025-08-01 02:00:00|   115600.0|     NULL|   NULL|
| 1948905|2025-08-01 02:15:00|  115619.94|     NULL|   NULL|
| 1948906|2025-08-01 02:30:00|  115900.01|     NULL|   NULL|
| 1948907|2025-08-01 02:45:00|  115966.12|     NULL|   NULL|
| 1948908|2025-08-01 03:00:00|   115816.0|     NULL|   NULL|
| 1948909|2025-08-01 03:

In [21]:
spark.sql("""
select * from serving_db.pattern_three where pattern is not null
""").show()

+--------+----------+-----------+-----+-------+
|group_id|group_date|close_price|trend|pattern|
+--------+----------+-----------+-----+-------+
+--------+----------+-----------+-----+-------+

